# Solving Poisson with scikit-fem on a FieldML Mesh

This notebook takes the bundled `unit_cube` FieldML document, converts it to a `skfem.Mesh` + `skfem.Basis` pair, solves the Poisson problem

$$-\nabla^2 u = 1 \text{ in } \Omega, \qquad u = 0 \text{ on } \partial\Omega,$$

and visualizes the field, its gradient magnitude, and contour slices in a four-panel pyvista figure. It demonstrates that pyfieldml slots directly into a real FEM workflow with no custom adapters.

## Install scikit-fem

```bash
pip install pyfieldml[scikit-fem]
```

In [ ]:
import numpy as np

from pyfieldml import datasets
from pyfieldml.interop.scikit_fem import to_scikit_fem

## Load a FieldML document and hand it to scikit-fem

In [ ]:
doc = datasets.load_unit_cube()
mesh, basis = to_scikit_fem(doc)
print("skfem mesh:", type(mesh).__name__)
print("skfem basis:", type(basis).__name__)
print("dofs :", basis.N)

## Assemble and solve on a refined mesh

A textbook Poisson assembly on the unit cube. Homogeneous Dirichlet BCs on every boundary face; unit source. We refine the FieldML unit cube a few times so there is a visible interior field to plot.

In [ ]:
from skfem import Basis, BilinearForm, LinearForm, condense, solve
from skfem.helpers import dot, grad


@BilinearForm
def laplace(u, v, _):
    return dot(grad(u), grad(v))


@LinearForm
def source(v, _):
    return 1.0 * v


mesh_r = mesh.refined(3)
basis_r = Basis(mesh_r, basis.elem)
A = laplace.assemble(basis_r)
b = source.assemble(basis_r)
dirichlet = basis_r.get_dofs()
u = solve(*condense(A, b, D=dirichlet))
print("dofs after refinement:", basis_r.N)
print("solved; max(u) =", float(u.max()))

## Convergence study

Sweep refinement levels and track the peak solution value. For the unit-cube Poisson problem this converges to the analytic solution max ~ 0.0559 (Dirichlet-Laplacian eigen-expansion).

In [ ]:
import matplotlib.pyplot as plt

levels = range(1, 5)
peaks = []
dofs = []
for lv in levels:
    m_lv = mesh.refined(lv)
    b_lv = Basis(m_lv, basis.elem)
    A_lv = laplace.assemble(b_lv)
    f_lv = source.assemble(b_lv)
    u_lv = solve(*condense(A_lv, f_lv, D=b_lv.get_dofs()))
    peaks.append(float(u_lv.max()))
    dofs.append(b_lv.N)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogx(dofs, peaks, marker="o")
ax.axhline(0.0559, color="gray", linestyle=":", label="analytic ~ 0.0559")
ax.set_xlabel("degrees of freedom")
ax.set_ylabel("max u")
ax.set_title("Poisson on unit cube: peak solution vs refinement")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()

## Four-panel solution visualization

1. wireframe mesh; 2. scalar u colored on the volume; 3. gradient magnitude on a clipped surface; 4. iso-contours.

In [ ]:
import pyvista as pv

pv.OFF_SCREEN = True
pv.set_jupyter_backend("static")

pts = mesh_r.p.T
cells_t = mesh_r.t.T
n_v = cells_t.shape[1]
VTK_CELL = {4: 10, 8: 12, 3: 5, 6: 13}  # tet=10, hex=12, triangle=5, wedge=13
cells = np.hstack([np.full((cells_t.shape[0], 1), n_v, dtype=np.int64), cells_t]).astype(np.int64)
grid = pv.UnstructuredGrid(
    cells,
    np.full(cells_t.shape[0], VTK_CELL[n_v]),
    pts,
)
grid.point_data["u"] = u

# Approximate gradient magnitude at nodes via pyvista filter
grad_grid = grid.compute_derivative(scalars="u", gradient=True)
grad_grid["|grad u|"] = np.linalg.norm(grad_grid["gradient"], axis=1)

p = pv.Plotter(off_screen=True, window_size=(1000, 850), shape=(2, 2))
p.subplot(0, 0)
p.add_mesh(grid, style="wireframe", color="steelblue", line_width=1)
p.add_text("refined mesh", font_size=10)
p.view_isometric()
p.subplot(0, 1)
p.add_mesh(grid, scalars="u", cmap="viridis", show_edges=False, scalar_bar_args={"title": "u"})
p.add_text("scalar field u", font_size=10)
p.view_isometric()
p.subplot(1, 0)
clipped = grad_grid.clip(normal="x", origin=grad_grid.center, invert=False)
p.add_mesh(
    clipped,
    scalars="|grad u|",
    cmap="magma",
    show_edges=False,
    scalar_bar_args={"title": "|grad u|"},
)
p.add_text("|grad u| (clipped)", font_size=10)
p.view_isometric()
p.subplot(1, 1)
contours = grid.contour(isosurfaces=6, scalars="u")
p.add_mesh(grid, color="lightgray", opacity=0.15)
p.add_mesh(contours, cmap="viridis", scalars="u", show_edges=False)
p.add_text("iso-contours of u", font_size=10)
p.view_isometric()
p.show(screenshot="/tmp/poisson4panel.png")

## Inspect the solution at a few interior DOFs

In [ ]:
dof_coords = basis_r.doflocs.T
interior_mask = np.ones(basis_r.N, dtype=bool)
interior_mask[basis_r.get_dofs()] = False
print("interior DOF count:", int(interior_mask.sum()))
print("interior solution samples:")
for i in np.flatnonzero(interior_mask)[:5]:
    print(f"  x={dof_coords[i]}  u={u[i]:.4f}")

### Takeaway

Any FieldML Lagrange mesh is now one function call away from the full scikit-fem stack - assembly, quadrature, boundary traces, adaptive refinement. See `pyfieldml.interop.scikit_fem` for the supported topology + order table.